# Laboratorio 02 — Análisis exploratorio (Nivel B)

Este notebook **solo** importa funciones de `src/` y genera figuras; no define
indicadores, el motor de backtesting, ni la lógica de optimización (esa vive
exclusivamente en `src/`). Tiene dos partes:

1. Un recorrido **ilustrativo y rápido** (auditoría de datos, indicadores,
   una muestra pequeña del walk-forward) para explicar cómo funciona cada
   pieza.
2. La **lectura de los resultados completos y definitivos**, que ya fueron
   calculados por `python main.py` y quedaron guardados en `docs/tablas/` y
   `docs/figuras/`. Los números que importan para el reporte son esos, no
   los de la sección ilustrativa.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import json
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from src import config, data, signals, backtest, metrics, regimes, optimize, plots

pd.set_option("display.max_columns", 40)

## 1. Auditoría de datos (src.data)

In [2]:
train, test, audit_train, audit_test = data.load_train_test()
print(audit_train.resumen_texto())
print()
print(audit_test.resumen_texto())

Auditoría de btc_project_train.csv
  Filas: 166,450 | Rango: 2022-06-01 00:00:00 -> 2023-12-31 00:00:00
  Gmtoffset único: [0] (0 = UTC, confirmado)
  Barras con OHLC vacío: 3,745 en 272 rachas (racha máxima: 270 barras)
  Marcas fuera de rejilla de 5 min: 573
  Barras planas (O=H=L=C): 3,355
  Volumen vacío: 47.5%
  Huecos grandes de calendario (> 30 min): 2
    - 2023-03-12 01:55:00 -> 2023-03-12 03:00:00 (1.08 h)
    - 2023-03-25 23:55:00 -> 2023-03-28 00:00:00 (48.08 h)

Auditoría de btc_project_test.csv
  Filas: 9,387 | Rango: 2023-12-31 00:00:00 -> 2024-06-03 03:27:00
  Gmtoffset único: [0] (0 = UTC, confirmado)
  Barras con OHLC vacío: 17 en 9 rachas (racha máxima: 4 barras)
  Marcas fuera de rejilla de 5 min: 3
  Barras planas (O=H=L=C): 29
  Volumen vacío: 51.6%
  Huecos grandes de calendario (> 30 min): 1
    - 2023-12-31 23:55:00 -> 2024-05-02 13:30:00 (2941.58 h)


In [3]:
display(data.episode_summary(train))
display(data.episode_summary(test))

,episode_id,inicio,fin,n_barras,n_barras_operables
0,1,2022-06-01 00:00:00,2023-03-12 01:55:00,80502,78486
1,2,2023-03-12 03:00:00,2023-03-25 23:55:00,3711,1695
2,3,2023-03-28 00:00:00,2023-12-31 00:00:00,78492,76476


,episode_id,inicio,fin,n_barras,n_barras_operables
0,1,2023-12-31 00:05:00,2023-12-31 23:55:00,288,0
1,2,2024-05-02 13:30:00,2024-06-03 03:27:00,9081,7065


## 2. Indicadores y regla de confirmación (src.signals)

Se calcula la señal confirmada sobre un tramo corto del episodio más largo
del train, solo para visualizar cómo se ven los tres indicadores y la
confirmación 2 de 3 sobre el precio.

In [4]:
episodio_largo = train[train["episode_id"] == train["episode_id"].value_counts().idxmax()].reset_index(drop=True)
muestra = episodio_largo.iloc[3000:5000].reset_index(drop=True)

parametros_demo = signals.StrategyParams(
    ema_fast=12, ema_slow=96, trend_band_k=0.2,
    rsi_window=14, rsi_band=10.0,
    boll_window=24, boll_k=2.2,
    atr_window=14, sl_atr_mult=2.0, tp_atr_mult=3.0,
    position_fraction=0.3,
)
tabla_demo = signals.compute_indicator_table(muestra, parametros_demo)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(muestra["Datetime"], muestra["Close"], color="black", linewidth=0.8, label="Close")
largos = tabla_demo["signal"] == 1
cortos = tabla_demo["signal"] == -1
ax.scatter(muestra.loc[largos, "Datetime"], muestra.loc[largos, "Close"], color="green", s=12, label="Señal larga (2 de 3)")
ax.scatter(muestra.loc[cortos, "Datetime"], muestra.loc[cortos, "Close"], color="red", s=12, label="Señal corta (2 de 3)")
ax.set_title("Señales de la estrategia sobre una muestra de train")
ax.set_xlabel("Fecha")
ax.set_ylabel("Precio")
ax.legend()
plt.show()

## 3. Muestra rápida del walk-forward (ilustrativa, pocas ventanas)

**Esto NO reemplaza la corrida completa de `python main.py`** (76 ventanas,
~15-25 minutos). Aquí solo se corren unas pocas ventanas para mostrar cómo
funciona la optimización híbrida Random Search + TPE y la detección de
régimen, en un tiempo razonable dentro del notebook.

In [5]:
wf_demo = optimize.run_walkforward(train, seed=config.RANDOM_SEED, max_windows=4, verbose=True)
wf_demo.windows_table[["window", "regime", "n_trials", "selection_method", "n_trades_is", "calmar_is", "n_trades_oos", "calmar_oos", "wfe"]]

  ventana 1/4: 2022-06-01 -> 2022-07-08 | configs acumuladas=360


  ventana 2/4: 2022-06-08 -> 2022-07-15 | configs acumuladas=720


  ventana 3/4: 2022-06-15 -> 2022-07-22 | configs acumuladas=1080


  ventana 4/4: 2022-06-22 -> 2022-07-29 | configs acumuladas=1440


,window,regime,n_trials,selection_method,n_trades_is,calmar_is,n_trades_oos,calmar_oos,wfe
0,0,tendencia,120,meseta,15,68.487294,10,-29.821866,-0.435436
1,0,reversion,120,meseta,21,14.551509,16,35.134194,2.414471
2,0,crisis,120,meseta,18,3015.195963,0,NaN,NaN
3,1,tendencia,120,meseta,30,32.177538,5,-15.913593,-0.494556
4,1,reversion,120,meseta,10,22.920159,4,-2.922672,-0.127515
5,1,crisis,120,meseta,30,-4.889009,0,NaN,NaN
6,2,tendencia,120,meseta,37,34.234141,22,-25.628211,-0.748616
7,2,reversion,120,meseta,23,77.473054,0,NaN,NaN
8,2,crisis,120,meseta,24,14.346200,0,NaN,NaN
9,3,tendencia,120,meseta,30,-10.210160,9,22.581944,NaN


In [6]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(wf_demo.oos_equity.index, wf_demo.oos_equity.values, color="#1f6feb")
ax.set_title("Curva de equity OOS - muestra de 4 ventanas de walk-forward")
ax.set_xlabel("Fecha")
ax.set_ylabel("Valor del portafolio (USD)")
plt.show()

## 4. Resultados completos (generados por `python main.py`)

A partir de aquí se leen los archivos reales en `docs/tablas/`, producidos
por la corrida completa del walk-forward (76 ventanas) y de la evaluación
en test. Estos son los números que se reportan en `docs/reporte.pdf`.

In [7]:
tablas_dir = config.TABLES_DIR
metricas = pd.read_csv(tablas_dir / "metricas_resumen.csv")
metricas

,Unnamed: 0,conjunto,sharpe,sortino,calmar,max_drawdown,win_rate,retorno_anualizado,volatilidad_anualizada,n_operaciones,payoff_ratio,retorno_total,benchmark_retorno_total,benchmark_calmar,benchmark_max_drawdown
0,0,train_oos_walkforward,-7.285516,-3.929484,-0.736334,0.505758,0.282784,-0.372407,0.068287,1365,1.279548,-0.501861,0.555298,0.906493,0.378899
1,1,test_completo,-5.988529,-3.292863,-1.882004,0.025598,0.190476,-0.048176,0.039196,42,1.565583,-0.020768,0.162614,5.400679,0.078777
2,2,test_segmento_dic2023,-27.013802,-2.251180,-316.960485,0.000829,0.000000,-0.262685,0.011239,1,NaN,-0.000829,NaN,NaN,NaN
3,3,test_segmento_may2024,-5.849056,-3.270890,-8.118594,0.025598,0.195122,-0.207822,0.039763,41,1.558828,-0.019956,NaN,NaN,NaN


In [8]:
ventanas = pd.read_csv(tablas_dir / "walkforward_ventanas.csv")
print(f"Ventanas optimizadas: {ventanas['window'].nunique()}")
print(f"WFE media (Calmar_IS > 0.05): {ventanas['wfe'].dropna().mean():.3f}")
ventanas.groupby("regime")[["calmar_is", "calmar_oos", "wfe", "n_trades_is", "n_trades_oos"]].mean()

Ventanas optimizadas: 74
WFE media (Calmar_IS > 0.05): 57481.507


,calmar_is,calmar_oos,wfe,n_trades_is,n_trades_oos
regime,,,,,
crisis,100.737545,1.185749e+04,-1.257419,18.716216,5.256757
reversion,6.710100,2.697167e+02,-2.492699,21.621622,5.648649
tendencia,355.134685,1.232291e+06,85296.147062,20.716216,7.500000


In [9]:
sensibilidad = pd.read_csv(tablas_dir / "sensibilidad_parametros.csv")
sensibilidad.head(15)

,Unnamed: 0,regimen,parametro,variacion,calmar,retorno_total,n_operaciones
0,0,base,ninguno,0.0,-0.741387,-0.538570,1291
1,1,tendencia,ema_fast,+20%,-0.736975,-0.520205,1276
2,2,tendencia,ema_fast,-20%,-0.743177,-0.545918,1308
3,3,tendencia,ema_slow,+20%,-0.734204,-0.508503,1276
4,4,tendencia,ema_slow,-20%,-0.746232,-0.558325,1303
5,5,tendencia,trend_band_k,+20%,-0.740267,-0.533938,1283
6,6,tendencia,trend_band_k,-20%,-0.746387,-0.558950,1306
7,7,tendencia,rsi_window,+20%,-0.729937,-0.493558,1207
8,8,tendencia,rsi_window,-20%,-0.752845,-0.578486,1405
9,9,tendencia,rsi_band,+20%,-0.712684,-0.418884,1135


In [10]:
costos = pd.read_csv(tablas_dir / "sensibilidad_costos.csv")
costos

,Unnamed: 0,comision,retorno_total,calmar,n_operaciones
0,0,0.000000,0.328909,3.422336,1291
1,1,0.000156,0.164474,1.151348,1291
2,2,0.000312,0.020409,0.103366,1291
3,3,0.000469,-0.106570,-0.430330,1291
4,4,0.000625,-0.217068,-0.601836,1291
5,5,0.000781,-0.313884,-0.672192,1291
6,6,0.000938,-0.399224,-0.706448,1291
7,7,0.001094,-0.473493,-0.725015,1291
8,8,0.001250,-0.538570,-0.741387,1291
9,9,0.002500,-0.839554,-0.840013,1291


In [11]:
comparacion = pd.read_csv(tablas_dir / "comparacion_confirmacion_vs_individual.csv")
comparacion

,Unnamed: 0,variante,n_operaciones,calmar,retorno_total
0,0,confirmacion_2_de_3,1291,-0.741387,-0.538570
1,1,solo_tendencia,2989,-0.845193,-0.849945
2,2,solo_momento,1533,-0.805351,-0.764285
3,3,solo_volatilidad,851,-0.696596,-0.333290


In [12]:
regimen_duracion_train = pd.read_csv(tablas_dir / "regimen_duracion_train.csv")
regimen_duracion_test = pd.read_csv(tablas_dir / "regimen_duracion_test.csv")
regimen_proporcion = pd.read_csv(tablas_dir / "regimen_proporcion_tiempo.csv")
display(regimen_duracion_train)
display(regimen_duracion_test)
display(regimen_proporcion)

,Unnamed: 0,regimen,duracion_media_horas,n_episodios,n_transiciones_total,regimen_nombre
0,0,0,80.435606,66,177,tendencia
1,1,1,52.985556,75,177,reversion
2,2,2,79.207207,37,177,crisis


,Unnamed: 0,regimen,duracion_media_horas,n_episodios,n_transiciones_total,regimen_nombre
0,0,0,4.666667,3,9,tendencia
1,1,1,133.687500,4,9,reversion
2,2,2,20.000000,3,9,crisis


,Unnamed: 0,train,test
0,tendencia,0.434668,0.022998
1,reversion,0.325375,0.878439
2,crisis,0.239956,0.098563


In [13]:
with open(tablas_dir / "friccion_ejecucion_test.json", encoding="utf-8") as f:
    friccion = json.load(f)
friccion

{'half_spread_frac': 0.00013092015613807992,
 'sigma_bar': 0.004200112235972216,
 'pnl_original': -20767.891800973357,
 'n_operaciones': 42,
 'costo_friccion_min': 1561.982654318373,
 'costo_friccion_max': 1775.8739204807302,
 'pnl_con_friccion_min': -22543.76572145409,
 'pnl_con_friccion_max': -22329.87445529173}

In [14]:
with open(tablas_dir / "parametros_congelados.json", encoding="utf-8") as f:
    congelado = json.load(f)
congelado["params_by_regime"]

{'tendencia': {'ema_fast': 40,
  'ema_slow': 191,
  'trend_band_k': 0.8218900594397459,
  'rsi_window': 10,
  'rsi_band': 24.34791300692887,
  'boll_window': 32,
  'boll_k': 3.7633208439978114,
  'atr_window': 7,
  'sl_atr_mult': 4.061996359033321,
  'tp_atr_mult': 11.208513750939227,
  'position_fraction': 0.5477531132647657},
 'reversion': {'ema_fast': 40,
  'ema_slow': 165,
  'trend_band_k': 0.6188054118795108,
  'rsi_window': 30,
  'rsi_band': 19.678566971103262,
  'boll_window': 27,
  'boll_k': 3.9655521405750624,
  'atr_window': 28,
  'sl_atr_mult': 4.780371931302418,
  'tp_atr_mult': 11.013640422860767,
  'position_fraction': 0.13198679986063697},
 'crisis': {'ema_fast': 33,
  'ema_slow': 194,
  'trend_band_k': 0.4215087144237473,
  'rsi_window': 20,
  'rsi_band': 27.95280877782957,
  'boll_window': 60,
  'boll_k': 2.092127110567116,
  'atr_window': 23,
  'sl_atr_mult': 4.933927495955989,
  'tp_atr_mult': 11.717004009775128,
  'position_fraction': 0.11278898689699472}}

## 5. Figuras

Todas las figuras obligatorias y de diagnóstico ya están generadas en
`docs/figuras/` por `python main.py`. Se listan a continuación.

In [15]:
for p in sorted(config.FIGURES_DIR.glob("*.png")):
    print(p.name)

fig01a_equity_train.png
fig01b_equity_test.png
fig02a_drawdown_train.png
fig02b_drawdown_test.png
fig03a_retornos_mensuales_train.png
fig03b_retornos_mensuales_test.png
fig04_sensibilidad_parametros.png
fig05_retorno_vs_costo.png
fig06a_regimen_timeline_train.png
fig06b_distribuciones_regimen.png
fig06c_equity_regimenes.png
fig07_historia_crisis.png
fig07_historia_reversion.png
fig07_historia_tendencia.png
fig08_importancia_crisis.png
fig08_importancia_reversion.png
fig08_importancia_tendencia.png
fig09_slice_crisis.png
fig09_slice_reversion.png
fig09_slice_tendencia.png
fig10_superficie_crisis.png
fig10_superficie_reversion.png
fig10_superficie_tendencia.png
